In [12]:
import numpy as np
import pandas as pd
print("pandas version:", pd.__version__)

pandas version: 3.0.2


In [13]:
df = pd.read_csv('../datasets/beneficiaries001.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn info:")
print(df.info())
print("\nBasic statistics:")
print(df.describe())
print(df.head(10))        # first 10 rows
print(df.tail(10))        # last 10 rows
print(df.sample(10))      # random 10 rows

Shape: (12, 10)

First 5 rows:
   beneficiary_id       full_name   age  gender  province  district  \
0               1      Ahmad Shah  45.0    Male     Kabul     Kabul   
1               2    fatima noori  32.0  female     kabul     Kabul   
2               3    Mohammad Ali   NaN    Male  Kandahar  Kandahar   
3               4      Ahmad Shah  45.0    Male     Kabul     Kabul   
4               5  Zainab Hussain  28.0       F     Herat     Herat   

   household_size  vulnerability_score registration_date        phone  
0             6.0                  7.5        2024-01-15  701234567.0  
1             4.0                  8.2        2024-01-15  709876543.0  
2             3.0                  6.1        2024-01-16          NaN  
3             6.0                  7.5        2024-01-15  701234567.0  
4             5.0                  9.0        2024-01-16  791234567.0  

Column info:
<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 10 columns):
 #  

## Data Quality Assessment 

In [14]:
# Missing values
print("=== Missing Values ===")
print(df.isnull().sum())
print("\nMissing percentage:")
print(df.isnull().sum() / len(df) * 100)

# Duplicates
print("\n=== Duplicates ===")
print("Total duplicate rows:", df.duplicated().sum())
print(df[df.duplicated(keep=False)])

# Unique values in categorical columns
print("\n=== Gender unique values ===")
print(df['gender'].value_counts())

print("\n=== Province unique values ===")
print(df['province'].value_counts())

# Impossible values
print("\n=== Age range ===")
print("Min age:", df['age'].min())
print("Max age:", df['age'].max())

print("\n=== Vulnerability score range ===")
print("Min:", df['vulnerability_score'].min())
print("Max:", df['vulnerability_score'].max())

=== Missing Values ===
beneficiary_id         0
full_name              1
age                    1
gender                 0
province               0
district               0
household_size         1
vulnerability_score    0
registration_date      0
phone                  2
dtype: int64

Missing percentage:
beneficiary_id          0.000000
full_name               8.333333
age                     8.333333
gender                  0.000000
province                0.000000
district                0.000000
household_size          8.333333
vulnerability_score     0.000000
registration_date       0.000000
phone                  16.666667
dtype: float64

=== Duplicates ===
Total duplicate rows: 0
Empty DataFrame
Columns: [beneficiary_id, full_name, age, gender, province, district, household_size, vulnerability_score, registration_date, phone]
Index: []

=== Gender unique values ===
gender
Male      7
female    2
F         1
FEMALE    1
Female    1
Name: count, dtype: int64

=== Province unique v

In [15]:
print(df.duplicated(subset='age',keep=False).sum())
print(df[df.duplicated(subset=['full_name', 'age', 'province'], keep=False)])

# Check 1 - strict match
name_age_province = df.duplicated(subset=['full_name', 'age', 'province'], keep=False).sum()

# Check 2 - phone based
phone_dupes = df.duplicated(subset=['phone'], keep=False).sum()

print("Name+Age+Province duplicates:", name_age_province)
print("Phone duplicates:", phone_dupes)

# Only flag phone duplicates where phone is NOT missing
phone_real_dupes = df[
    df.duplicated(subset=['phone'], keep=False) & 
    df['phone'].notna()
]
print(phone_real_dupes[['beneficiary_id', 'full_name', 'age', 'province', 'phone']])

3
    beneficiary_id   full_name   age gender province district  household_size  \
0                1  Ahmad Shah  45.0   Male    Kabul    Kabul             6.0   
3                4  Ahmad Shah  45.0   Male    Kabul    Kabul             6.0   
10              11  Ahmad Shah  45.0   Male    Kabul    Kabul             6.0   

    vulnerability_score registration_date        phone  
0                   7.5        2024-01-15  701234567.0  
3                   7.5        2024-01-15  701234567.0  
10                  7.5        2024-01-15  701234567.0  
Name+Age+Province duplicates: 3
Phone duplicates: 5
    beneficiary_id   full_name   age province        phone
0                1  Ahmad Shah  45.0    Kabul  701234567.0
3                4  Ahmad Shah  45.0    Kabul  701234567.0
10              11  Ahmad Shah  45.0    Kabul  701234567.0


## Data Cleaning

In [16]:
# Create clean copy
df_clean = df.copy()

# Check shape before
print("Before removing duplicates:", df_clean.shape)

# Remove duplicates
df_clean = df_clean.drop_duplicates(
    subset=['full_name', 'age', 'province'], 
    keep='first'
)

# Check shape after
print("After removing duplicates:", df_clean.shape)

# Confirm Ahmad Shah now appears only once
print("\nAhmad Shah rows remaining:")
print(df_clean[df_clean['full_name'] == 'Ahmad Shah'])

Before removing duplicates: (12, 10)
After removing duplicates: (10, 10)

Ahmad Shah rows remaining:
   beneficiary_id   full_name   age gender province district  household_size  \
0               1  Ahmad Shah  45.0   Male    Kabul    Kabul             6.0   

   vulnerability_score registration_date        phone  
0                  7.5        2024-01-15  701234567.0  


#### Fixing name casing

In [17]:
# Fix full_name to title case
df_clean['full_name'] = df_clean['full_name'].str.title()

# Verify
print(df_clean['full_name'].unique())

<StringArray>
[    'Ahmad Shah',   'Fatima Noori',   'Mohammad Ali', 'Zainab Hussain',
   'Abdul Rahman',              nan, 'Mariam Sultani',  'Khalid Ahmadi',
   'Sadia Rahimi',  'Najiba Karimi']
Length: 10, dtype: str


#### Fixing gender inconsistency
.str.replace() — finds and replaces text patterns inside a string, like find "abc" inside "xabcx"

.replace() — replaces an entire cell value, like replace the whole value "F" with "Female"

In [18]:
df_clean['gender'] = df_clean['gender'].str.title()
df_clean['gender'] = df_clean['gender'].replace('F', 'Female')
print(df_clean['gender'].value_counts())

gender
Male      5
Female    5
Name: count, dtype: int64


#### Fixing province casing

In [19]:
df_clean['province'] = df_clean['province'].str.title()
df_clean['district'] = df_clean['district'].str.title()

print(df_clean['province'].unique())

<StringArray>
['Kabul', 'Kandahar', 'Herat', 'Balkh']
Length: 4, dtype: str


#### Fixing Impossible Values
Impossible values are values that exist in the data but cannot be real. Age 999 is the classic example. These are different from missing values — the cell is not empty, it just contains nonsense.  
Two ways to handle impossible values:  
Option 1 — Replace with NaN:  
Convert the impossible value to missing. This is the professional approach because you're honestly saying "we don't know this value" rather than inventing something.  
Option 2 — Replace with a calculated value:  
Replace with mean or median of the column. This is called imputation.

In [20]:
# Replace impossible age with NaN
df_clean['age'] = df_clean['age'].replace(999, np.nan)

# Confirm
print("Max age after fix:", df_clean['age'].max())
print("\nAge column:")
print(df_clean['age'])

Max age after fix: 45.0

Age column:
0     45.0
1     32.0
2      NaN
4     28.0
5      NaN
6     35.0
7     29.0
8     38.0
9     24.0
11    31.0
Name: age, dtype: float64


#### Fixing vulnerability score above 10

In [21]:
# Replace any score above 10 with NaN
df_clean.loc[df_clean['vulnerability_score'] > 10, 'vulnerability_score'] = np.nan

# Confirm
print("Max vulnerability score:", df_clean['vulnerability_score'].max())

Max vulnerability score: 9.0


####  Handling Missing Values
Three options for handling missing values:  
Option 1 — Drop the row:  
Remove the entire row that has missing value. Simple but you lose data.  
Option 2 — Fill with a calculated value (Imputation):  
Replace missing value with mean, median, or mode of that column.  
Option 3 — Fill with a placeholder:  
Replace with "Unknown" or 0 depending on context.

In [22]:
df_clean['full_name'] = df_clean['full_name'].fillna('Unknown')
print(df_clean['full_name'].isnull().sum())

0


In [23]:
age_median = df_clean['age'].median()
print("Age median:", age_median)
df_clean['age'] = df_clean['age'].fillna(age_median)
print(df_clean['age'].isnull().sum())

Age median: 31.5
0
